# Explainable Industrial Defect Inspector — Colab Setup

Mounts Drive, clones the repo, installs dependencies, and creates the project folder structure.

**Workflow:** code locally → push to GitHub → run this notebook to test.

- Data lives in Drive: `defect_inspector/data/mvtec`
- Results live in Drive: `defect_inspector/outputs/`
- Code is cloned from GitHub to `/content/xAI_Defects`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/defect_inspector"
REPO_URL  = "https://github.com/AKIF-jk/xAI_Defects.git"
LOCAL_DIR = "/content/xAI_Defects"

# Mount point for Drive data
DRIVE_DATA    = os.path.join(DRIVE_ROOT, "data")
DRIVE_OUTPUTS = os.path.join(DRIVE_ROOT, "outputs")

os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Drive root: {DRIVE_ROOT}")
print(f"Exists: {os.path.exists(DRIVE_ROOT)}")

In [ ]:
%cd /content
if os.path.exists(LOCAL_DIR):
    %cd {LOCAL_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {LOCAL_DIR}

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet
!pip install open_clip_torch timm captum shap albumentations fastapi uvicorn python-multipart gradio faiss-cpu anthropic matplotlib seaborn scikit-learn --quiet

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name:      {torch.cuda.get_device_name(0)}")
    print(f"GPU count:     {torch.cuda.device_count()}")

In [ ]:
import open_clip, captum, shap, gradio
print(f"torch:              {torch.__version__}")
print(f"open_clip_torch:    {open_clip.__version__}")
print(f"captum:             {captum.__version__}")
print(f"shap:               {shap.__version__}")
print(f"gradio:             {gradio.__version__}")

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/defect_inspector"

folders = [
    "data/mvtec",
    "src/data", "src/model", "src/xai", "src/api",
    "outputs/heatmaps", "outputs/shap", "outputs/results",
]

for f in folders:
    os.makedirs(os.path.join(DRIVE_ROOT, f), exist_ok=True)

print("Drive folder structure created:")
for f in folders:
    path = os.path.join(DRIVE_ROOT, f)
    print(f"  {path}")

In [ ]:
import sys
SRC_PATH = os.path.join(DRIVE_ROOT, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

LOCAL_SRC = "/content/xAI_Defects/src"
if os.path.exists(LOCAL_SRC) and LOCAL_SRC not in sys.path:
    sys.path.insert(0, LOCAL_SRC)

print(f"sys.path entries:\n  {SRC_PATH}\n  {LOCAL_SRC}")

In [ ]:
print("Environment ready")

## Part 2: MVTec AD Dataset

Download, extract, verify, and visualize the 15-category benchmark.

Archive is cached in Drive so this is a one-time cost.

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/defect_inspector"
MVTEC_DIR = os.path.join(DRIVE_ROOT, "data", "mvtec")
ARCHIVE = os.path.join(MVTEC_DIR, "mvtec_anomaly_detection.tar.xz")
EXTRACTED = os.path.join(MVTEC_DIR, "mvtec_anomaly_detection")
os.makedirs(MVTEC_DIR, exist_ok=True)

URL = "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420938113-1629952094/mvtec_anomaly_detection.tar.xz"

if os.path.exists(EXTRACTED):
    print(f"MVTec AD already extracted at {EXTRACTED}")
elif not os.path.exists(ARCHIVE):
    print("Downloading MVTec AD (4.7 GB)...")
    import subprocess
    subprocess.run(["wget", URL, "-O", ARCHIVE, "--progress=bar:force", "--continue"], check=True)
    print("Download complete")
else:
    print("Archive already downloaded, skipping")

In [ ]:
import tarfile
from tqdm.notebook import tqdm

DRIVE_ROOT = "/content/drive/MyDrive/defect_inspector"
MVTEC_DIR = os.path.join(DRIVE_ROOT, "data", "mvtec")
ARCHIVE = os.path.join(MVTEC_DIR, "mvtec_anomaly_detection.tar.xz")
EXTRACTED = os.path.join(MVTEC_DIR, "mvtec_anomaly_detection")

if not os.path.exists(EXTRACTED):
    print("Extracting MVTec AD...")
    with tarfile.open(ARCHIVE, "r:xz") as tar:
        members = tar.getmembers()
        for member in tqdm(members, desc="Extracting"):
            tar.extract(member, path=MVTEC_DIR)
    print(f"Extraction complete: {len(members)} files extracted")
else:
    print("Already extracted")

In [ ]:
import os, glob

EXTRACTED = "/content/drive/MyDrive/defect_inspector/data/mvtec/mvtec_anomaly_detection"

CATEGORIES = ["bottle","cable","capsule","carpet","grid","hazelnut","leather",
              "metal_nut","pill","screw","tile","toothbrush","transistor","wood","zipper"]

found = [c for c in CATEGORIES if os.path.isdir(os.path.join(EXTRACTED, c))]
missing = [c for c in CATEGORIES if c not in found]
print(f"Categories found: {len(found)}/15")
if missing:
    print(f"Missing: {missing}")

print(f"\n{'Category':<15} {'Train':>6} {'Test':>6} {'Defect Types'}")
print("-" * 65)

category_stats = {}
for cat in CATEGORIES:
    cat_dir = os.path.join(EXTRACTED, cat)
    if not os.path.isdir(cat_dir):
        continue

    train_dir = os.path.join(cat_dir, "train", "good")
    train_count = len(glob.glob(os.path.join(train_dir, "*.png")))

    test_dir = os.path.join(cat_dir, "test")
    test_dirs = [d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))]
    defect_types = sorted(d for d in test_dirs if d != "good")
    test_anom = sum(len(glob.glob(os.path.join(test_dir, d, "*.png"))) for d in defect_types)
    test_good = len(glob.glob(os.path.join(test_dir, "good", "*.png")))
    test_count = test_good + test_anom

    print(f"{cat:<15} {train_count:>6} {test_count:>6} {', '.join(defect_types)}")
    category_stats[cat] = {"train": train_count, "normal_test": test_good,
                          "anom_test": test_anom, "defects": defect_types}

In [ ]:
import matplotlib.pyplot as plt

cats = list(category_stats.keys())
anom_counts = [category_stats[c]["anom_test"] for c in cats]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(cats)), anom_counts, color="crimson", edgecolor="black", linewidth=0.5)
ax.set_xticks(range(len(cats)))
ax.set_xticklabels(cats, rotation=45, ha="right")
ax.set_ylabel("Anomalous Test Images")
ax.set_title("MVTec AD: Anomalous Test Images per Category")
ax.grid(axis="y", alpha=0.3)

for bar, count in zip(bars, anom_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(count), ha="center", va="bottom", fontsize=9)

plt.tight_layout()

save_path = "/content/drive/MyDrive/defect_inspector/outputs/results/dataset_overview.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {save_path}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

EXTRACTED = "/content/drive/MyDrive/defect_inspector/data/mvtec/mvtec_anomaly_detection"
bottle_dir = os.path.join(EXTRACTED, "bottle")

test_dir = os.path.join(bottle_dir, "test")
defect_dirs = sorted(d for d in os.listdir(test_dir)
                     if d != "good" and os.path.isdir(os.path.join(test_dir, d)))
defect_type = defect_dirs[0]

normal_path = os.path.join(bottle_dir, "train", "good",
                           sorted(os.listdir(os.path.join(bottle_dir, "train", "good")))[0])
anom_path = os.path.join(bottle_dir, "test", defect_type,
                         sorted(os.listdir(os.path.join(bottle_dir, "test", defect_type)))[0])
mask_path = os.path.join(bottle_dir, "ground_truth", defect_type,
                         sorted(os.listdir(os.path.join(bottle_dir, "ground_truth", defect_type)))[0])

normal = plt.imread(normal_path)
anomalous = plt.imread(anom_path)
mask = plt.imread(mask_path)

if mask.ndim == 3:
    mask_gray = mask[:, :, 0]
else:
    mask_gray = mask

overlay = np.zeros((*mask_gray.shape, 4))
overlay[:, :, 0] = 1.0
overlay[:, :, 3] = mask_gray * 0.6

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(normal)
axes[0].set_title("Normal (good)")
axes[0].axis("off")

axes[1].imshow(anomalous)
axes[1].set_title(f"Anomalous ({defect_type})")
axes[1].axis("off")

axes[2].imshow(anomalous)
axes[2].imshow(overlay)
axes[2].set_title("Anomalous + Defect Mask")
axes[2].axis("off")

plt.tight_layout()
plt.show()
print(f"Normal: {normal_path}\nAnomalous: {anom_path}\nMask: {mask_path}")